# 08. Extract Handcrafted Audio Features

이 노트북에서는 `segment_manifest_10s.csv`의 10초 segment를 실제로 읽어
Logistic Regression / SVM baseline에 사용할 handcrafted audio feature를 추출한다.

## 사용 특징

- MFCC 40
- MFCC Δ
- MFCC Δ²
- Spectral centroid
- Spectral bandwidth
- Spectral rolloff
- Spectral flatness
- Spectral contrast
- RMS
- Zero-crossing rate (ZCR)

각 frame-level feature에서 **mean / std**를 계산하여 고정 길이 feature vector로 만든다.

## 오디오 설정

- Sample rate: **24,000 Hz**
- Segment length: **10초**
- Target samples: **240,000**
- `n_fft = 1024`
- `hop_length = 240`
- `n_mels = 128`
- `fmax = 12,000 Hz`

10초보다 몇 ms 짧은 segment는 이전 단계의 계획대로 zero-padding하여 정확히 10초로 맞춘다.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

FEATURE_DIR = PROJECT_ROOT / "data/processed/features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = FEATURE_DIR / "handcrafted_features_10s_checkpoint.pkl"
OUTPUT_PATH = FEATURE_DIR / "handcrafted_features_10s.csv"

SR = 24_000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)

N_FFT = 1024
HOP_LENGTH = 240
N_MELS = 128
N_MFCC = 40
FMAX = 12_000

CHECKPOINT_EVERY = 250

print("SEGMENT_PATH    :", SEGMENT_PATH)
print("OUTPUT_PATH     :", OUTPUT_PATH)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)
print("Target samples  :", TARGET_SAMPLES)


## 1. 라이브러리 확인

`librosa`를 이용해 waveform 로딩과 audio feature 추출을 수행한다.


In [ ]:
try:
    import librosa
    print("librosa version:", librosa.__version__)
except ImportError:
    raise ImportError(
        "librosa가 설치되어 있지 않습니다. "
        "이 노트북의 새 코드 셀에서 `%pip install librosa soundfile`을 실행한 뒤 커널을 다시 시작하세요."
    )


## 2. Segment Manifest 로드 및 기본 확인


In [ ]:
segments = pd.read_csv(SEGMENT_PATH)

print("===== SEGMENT MANIFEST =====")
print("Rows             :", len(segments))
print("Unique segment_id:", segments["segment_id"].nunique())
print("Unique tracks    :", segments["track_sample_id"].nunique())
print("Unique groups    :", segments["original_audio"].nunique())

print("\nSplit:")
print(segments["split"].value_counts())

print("\nLabel:")
print(segments["label"].value_counts())

display(segments.head())


## 3. 10초 Waveform 로딩 함수

각 segment의 `start_sec`부터 필요한 오디오를 읽는다.

- 10초보다 길면 정확히 240,000 samples로 자름
- 부족하면 뒤쪽을 0으로 padding
- stereo는 mono로 변환
- 모든 파일은 24 kHz로 resampling


In [ ]:
def load_segment_waveform(row):
    full_path = PROJECT_ROOT / row["audio_path"]

    if not full_path.exists():
        raise FileNotFoundError(full_path)

    y, _ = librosa.load(
        full_path,
        sr=SR,
        mono=True,
        offset=float(row["start_sec"]),
        duration=SEGMENT_SEC,
    )

    y = np.asarray(y, dtype=np.float32)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(
            y,
            (0, TARGET_SAMPLES - len(y)),
            mode="constant"
        )
    elif len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]

    if len(y) != TARGET_SAMPLES:
        raise RuntimeError(
            f"waveform length error: {len(y)} != {TARGET_SAMPLES}"
        )

    if not np.isfinite(y).all():
        raise ValueError("waveform contains NaN or Inf")

    return y


## 4. Feature 추출 함수

각 feature는 시간축에 따라 여러 frame을 가지므로
최종적으로 각 feature 채널의 **mean / std**를 계산한다.

예를 들어 MFCC는 40차원이므로:

```text
mfcc_01_mean
mfcc_01_std
...
mfcc_40_mean
mfcc_40_std
```

형태가 된다.

Δ와 Δ²도 동일하게 계산한다.


In [ ]:
def add_mean_std(feature_dict, prefix, x):
    x = np.asarray(x)

    if x.ndim == 1:
        x = x[np.newaxis, :]

    for i in range(x.shape[0]):
        name = (
            f"{prefix}_{i+1:02d}"
            if x.shape[0] > 1
            else prefix
        )

        feature_dict[f"{name}_mean"] = float(np.mean(x[i]))
        feature_dict[f"{name}_std"] = float(np.std(x[i]))


def extract_handcrafted_features(y):
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=FMAX,
    )

    mfcc_delta = librosa.feature.delta(mfcc, order=1)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    add_mean_std(features, "mfcc", mfcc)
    add_mean_std(features, "mfcc_delta", mfcc_delta)
    add_mean_std(features, "mfcc_delta2", mfcc_delta2)

    # Spectral features
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        roll_percent=0.85,
    )

    flatness = librosa.feature.spectral_flatness(
        y=y,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    rms = librosa.feature.rms(
        y=y,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH,
    )

    zcr = librosa.feature.zero_crossing_rate(
        y=y,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH,
    )

    add_mean_std(features, "spectral_centroid", centroid)
    add_mean_std(features, "spectral_bandwidth", bandwidth)
    add_mean_std(features, "spectral_rolloff", rolloff)
    add_mean_std(features, "spectral_flatness", flatness)
    add_mean_std(features, "spectral_contrast", contrast)
    add_mean_std(features, "rms", rms)
    add_mean_std(features, "zcr", zcr)

    return features


## 5. 한 Segment만 시험 추출

전체 10,077개를 처리하기 전에 첫 segment 하나가 정상적으로 로드되고
feature가 생성되는지 확인한다.


In [ ]:
test_row = segments.iloc[0]

test_y = load_segment_waveform(test_row)
test_features = extract_handcrafted_features(test_y)

print("Waveform shape :", test_y.shape)
print("Waveform dtype :", test_y.dtype)
print("Feature count  :", len(test_features))

print("\nFirst 10 features:")
for key in list(test_features.keys())[:10]:
    print(key, "=", test_features[key])


### 기대 feature 차원

현재 설정에서 feature 수는 다음과 같다.

- MFCC: 40 × mean/std = 80
- Δ: 40 × mean/std = 80
- Δ²: 40 × mean/std = 80
- centroid: 2
- bandwidth: 2
- rolloff: 2
- flatness: 2
- contrast: 7 × mean/std = 14
- RMS: 2
- ZCR: 2

총 **266개 feature**가 기대된다.


In [ ]:
EXPECTED_FEATURE_COUNT = 266

print("Expected features:", EXPECTED_FEATURE_COUNT)
print("Actual features  :", len(test_features))

assert len(test_features) == EXPECTED_FEATURE_COUNT, (
    f"Feature count mismatch: {len(test_features)}"
)


## 6. 전체 Segment Feature 추출

10,077개 segment를 순차적으로 처리한다.

시간이 오래 걸릴 수 있으므로:

- `CHECKPOINT_EVERY = 250` segment마다 checkpoint 저장
- 중간에 중단되어도 checkpoint가 있으면 이미 처리한 segment는 건너뜀
- 오류가 발생한 segment는 `error` 컬럼에 기록


In [ ]:
META_COLUMNS = [
    "segment_id",
    "track_sample_id",
    "original_audio",
    "label",
    "label_id",
    "source",
    "genre",
    "generator",
    "audio_path",
    "track_id",
    "split",
    "segment_index",
    "segment_role",
    "start_sec",
    "end_sec",
    "requires_padding",
    "pad_sec",
]

if CHECKPOINT_PATH.exists():
    feature_df = pd.read_pickle(CHECKPOINT_PATH)
    print("Loaded checkpoint rows:", len(feature_df))
else:
    feature_df = pd.DataFrame()
    print("No checkpoint found. Starting from zero.")

processed_ids = (
    set(feature_df["segment_id"].astype(str))
    if len(feature_df) and "segment_id" in feature_df.columns
    else set()
)

print("Already processed:", len(processed_ids))
print("Remaining        :", len(segments) - len(processed_ids))


In [ ]:
feature_rows = []

remaining = segments[
    ~segments["segment_id"].astype(str).isin(processed_ids)
]

for n, (_, row) in enumerate(remaining.iterrows(), start=1):
    result = {
        col: row[col]
        for col in META_COLUMNS
        if col in row.index
    }

    result["error"] = ""

    try:
        y = load_segment_waveform(row)
        feats = extract_handcrafted_features(y)

        result.update(feats)

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"

    feature_rows.append(result)

    if n % CHECKPOINT_EVERY == 0 or n == len(remaining):
        new_df = pd.DataFrame(feature_rows)

        if len(feature_df):
            combined = pd.concat(
                [feature_df, new_df],
                ignore_index=True,
            )
        else:
            combined = new_df.copy()

        combined = (
            combined
            .drop_duplicates("segment_id", keep="last")
            .reset_index(drop=True)
        )

        combined.to_pickle(CHECKPOINT_PATH)

        print(
            f"processed this run: {n}/{len(remaining)} | "
            f"checkpoint rows: {len(combined)}"
        )

        feature_df = combined
        feature_rows = []

print("\nExtraction finished.")
print("Rows:", len(feature_df))


## 7. 추출 오류 확인


In [ ]:
feature_df["error"] = feature_df["error"].fillna("")

failed = feature_df[
    feature_df["error"].str.len() > 0
].copy()

print("Feature extraction success:", len(feature_df) - len(failed))
print("Feature extraction failed :", len(failed))

if len(failed):
    display(
        failed[
            [
                "segment_id",
                "track_sample_id",
                "label",
                "generator",
                "audio_path",
                "error",
            ]
        ].head(30)
    )


## 8. Feature 컬럼 및 결측값 검사

metadata를 제외한 숫자 feature만 선택하여
NaN / Inf가 존재하는지 확인한다.


In [ ]:
FEATURE_COLUMNS = sorted([
    c for c in feature_df.columns
    if (
        c.startswith("mfcc_")
        or c.startswith("mfcc_delta_")
        or c.startswith("mfcc_delta2_")
        or c.startswith("spectral_")
        or c.startswith("rms_")
        or c.startswith("zcr_")
    )
])

print("Feature columns:", len(FEATURE_COLUMNS))

numeric_features = feature_df[FEATURE_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce"
)

nan_count = int(numeric_features.isna().sum().sum())
inf_count = int(
    np.isinf(numeric_features.to_numpy(dtype=float)).sum()
)

print("NaN values:", nan_count)
print("Inf values:", inf_count)

assert len(FEATURE_COLUMNS) == EXPECTED_FEATURE_COUNT


## 9. Segment Manifest와 정확히 대응하는지 검증

feature 결과의 segment ID가 원본 `segment_manifest_10s.csv`와
정확히 1:1로 일치하는지 확인한다.


In [ ]:
expected_ids = set(segments["segment_id"].astype(str))
feature_ids = set(feature_df["segment_id"].astype(str))

missing_ids = expected_ids - feature_ids
extra_ids = feature_ids - expected_ids

print("Expected segments:", len(expected_ids))
print("Feature rows     :", len(feature_ids))
print("Missing IDs      :", len(missing_ids))
print("Extra IDs        :", len(extra_ids))
print("Duplicate IDs    :", int(feature_df["segment_id"].duplicated().sum()))


## 10. Split / Label 분포 유지 확인


In [ ]:
print("===== FEATURE ROWS BY SPLIT =====")
print(feature_df["split"].value_counts())

print("\n===== FEATURE ROWS BY LABEL =====")
print(feature_df["label"].value_counts())

print("\n===== SPLIT × LABEL =====")
display(
    pd.crosstab(
        feature_df["split"],
        feature_df["label"]
    )
)


## 11. 간단한 Feature 통계 확인

극단적인 NaN/Inf뿐 아니라 feature 값의 범위가 대략 정상인지 확인한다.

StandardScaler는 다음 모델 단계에서 **Train 데이터에만 fit**한다.
이 단계에서는 scaling하지 않는다.


In [ ]:
feature_summary = numeric_features.describe().T

display(feature_summary.head(20))

print("\nFeature mean range:")
print(
    feature_summary["mean"].min(),
    "~",
    feature_summary["mean"].max()
)

print("\nFeature std range:")
print(
    feature_summary["std"].min(),
    "~",
    feature_summary["std"].max()
)


## 12. 최종 Feature QC


In [ ]:
qc_summary = pd.DataFrame({
    "check": [
        "segment_manifest_rows",
        "feature_rows",
        "unique_segment_id",
        "duplicate_segment_id",
        "feature_columns",
        "extraction_failures",
        "missing_segment_ids",
        "extra_segment_ids",
        "nan_feature_values",
        "inf_feature_values",
        "missing_split",
        "missing_label",
    ],
    "value": [
        len(segments),
        len(feature_df),
        feature_df["segment_id"].nunique(),
        int(feature_df["segment_id"].duplicated().sum()),
        len(FEATURE_COLUMNS),
        len(failed),
        len(missing_ids),
        len(extra_ids),
        nan_count,
        inf_count,
        int(feature_df["split"].isna().sum()),
        int(feature_df["label"].isna().sum()),
    ],
})

display(qc_summary)

core_qc_pass = (
    len(feature_df) == len(segments)
    and feature_df["segment_id"].nunique() == len(segments)
    and int(feature_df["segment_id"].duplicated().sum()) == 0
    and len(FEATURE_COLUMNS) == EXPECTED_FEATURE_COUNT
    and len(failed) == 0
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and nan_count == 0
    and inf_count == 0
    and int(feature_df["split"].isna().sum()) == 0
    and int(feature_df["label"].isna().sum()) == 0
)

print("===== FINAL RESULT =====")
print("Handcrafted Feature Core QC PASS:", core_qc_pass)


## 13. 최종 Feature CSV 저장

QC 통과 후 최종 feature table을 CSV로 저장한다.

`handcrafted_features_10s.csv`는 다음 단계의
Logistic Regression / RBF-SVM에서 직접 사용한다.


In [ ]:
if not core_qc_pass:
    raise RuntimeError(
        "Handcrafted feature QC가 통과하지 않았습니다. "
        "저장 전에 위 결과를 확인하세요."
    )

feature_df = feature_df.sort_values("segment_id").reset_index(drop=True)

feature_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)
print("Rows :", len(feature_df))
print("Features:", len(FEATURE_COLUMNS))


## 다음 단계

다음 단계에서는 같은 handcrafted feature를 사용해 두 baseline을 비교한다.

### Model 0 — Logistic Regression
- StandardScaler: **Train에서만 fit**
- class imbalance 대응
- Validation에서 threshold 선택
- Test에는 threshold 고정

### Model 1 — RBF SVM
- 동일한 266개 handcrafted feature
- StandardScaler 동일 원칙
- RBF kernel
- probability 또는 decision score 기반 평가

평가 지표:

- EER
- ROC-AUC
- PR-AUC
- Balanced Accuracy
- Macro-F1
- REAL FPR
- FAKE Miss Rate

그리고 segment-level 예측뿐 아니라
같은 `track_sample_id`의 segment 확률을 평균하여
**track-level 성능도 함께 평가**한다.
